# Phase 6 : Modélisation par Deep Learning (DL)

Ce notebook présente la modélisation profonde pour la prévision des arrivées touristiques au Maroc.

## Méthodologie et Architectures :
1. **Séparation & Mise à l'échelle propre** : Pour éviter toute fuite de données (*data leakage*), les paramètres du `MinMaxScaler` sont calculés uniquement sur l'ensemble d'entraînement, puis appliqués sur le test.
2. **Fenêtres Temporelles (Sequencing)** : Préparation des données sous forme de séquences temporelles de 12 mois de décalage.
3. **Recherche d'Architecture Optima (Optuna)** : Optimisation bayésienne des hyperparamètres (nombre d'unités, taux de dropout, learning rate) sur les réseaux récurrents : RNN, LSTM, GRU, Stacked LSTM et CNN-LSTM.
4. **Réseau Transformer** : Construction et évaluation d'un modèle basé sur les mécanismes d'attention (`MultiHeadAttention`).
5. **Comparaison Finale** : Bilan comparatif de tous les modèles ML et DL.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_squared_error

# Ajout du dossier parent au chemin système pour importer src
sys.path.append(os.path.abspath('..'))

# Importation des modules personnalisés propres
from src.config import DATA_DIR
import src.models_dl as dl
import src.evaluation as eval_mod
import src.visualization as viz

## 1. Chargement et Préparation des Séquences Temporelles

Nous chargeons le jeu de données final reconstruit et préparons les séquences à l'aide de notre module `src.models_dl`.

In [ ]:
df_full = pd.read_csv(os.path.join(DATA_DIR, 'merged_tourism_data_final.csv'))

# Préparation des séquences avec décalage de 12 mois et répartition 80% Train / 20% Test
X_train, X_test, y_train, y_test, scaler = dl.prepare_dl_sequences(df_full, window_size=12, split_ratio=0.8)

print(f"X_train shape: {X_train.shape} | y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape} | y_test shape: {y_test.shape}")

## 2. Optimisation d'Architecture RNN/LSTM/GRU avec Optuna

Nous lançons 15 essais pour trouver la meilleure architecture et les meilleurs hyperparamètres.

In [ ]:
print("Recherche automatique du meilleur modèle récurrent par Optuna...")
# Nous limitons l'entraînement à 20 époques par essai pour la vitesse de recherche
study = dl.optimize_dl_hyperparameters(X_train, y_train, X_test, y_test, n_trials=15, epochs=20)

print(f"\nMeilleure architecture trouvée : {study.best_params['model_type']}")
print(f"Meilleurs hyperparamètres : {study.best_params}")

## 3. Entraînement Final du Meilleur Modèle Optuna

Nous ré-entraînons le meilleur modèle trouvé par Optuna sur 50 époques.

In [ ]:
best_p = study.best_params
input_shape = (X_train.shape[1], X_train.shape[2])

best_model = dl.build_dl_model(
    model_type=best_p['model_type'],
    units=best_p['units'],
    dropout=best_p['dropout'],
    lr=best_p['lr'],
    input_shape=input_shape
)

print(f"Entraînement final du modèle {best_p['model_type']}...")
best_model.fit(X_train, y_train, epochs=50, batch_size=16, verbose=0)

# Prédictions et inversion de la mise à l'échelle (scaling)
y_pred_scaled = best_model.predict(X_test)

# Pour inverser le scaling de la cible uniquement, nous reconstruisons un tableau temporaire
dummy_train = np.zeros((len(y_pred_scaled), X_train.shape[2]))
dummy_train[:, 0] = y_pred_scaled.flatten()
y_pred_inv = scaler.inverse_transform(dummy_train)[:, 0]

dummy_test = np.zeros((len(y_test), X_train.shape[2]))
dummy_test[:, 0] = y_test
y_test_inv = scaler.inverse_transform(dummy_test)[:, 0]

# Métriques
best_dl_metrics = eval_mod.print_metrics_summary(y_test_inv, y_pred_inv, model_name=f"Best DL ({best_p['model_type']})")

## 4. Entraînement du Modèle Transformer (Multi-Head Attention)

Nous entraînons un modèle Transformer sur 50 époques.

In [ ]:
print("Entraînement du modèle Transformer...")
transformer_model = dl.build_transformer_model(input_shape)
transformer_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss="mse")

transformer_model.fit(X_train, y_train, epochs=50, batch_size=16, validation_split=0.1, verbose=0)

# Prédictions et inversion du scaling
y_pred_trans_scaled = transformer_model.predict(X_test)
dummy_trans = np.zeros((len(y_pred_trans_scaled), X_train.shape[2]))
dummy_trans[:, 0] = y_pred_trans_scaled.flatten()
y_pred_trans_inv = scaler.inverse_transform(dummy_trans)[:, 0]

# Métriques
trans_metrics = eval_mod.print_metrics_summary(y_test_inv, y_pred_trans_inv, model_name="Transformer")

## 5. Visualisation Graphique Comparatrice

Comparaison des prévisions ré-inversées sur l'échelle d'origine des arrivées.

In [ ]:
dl_predictions = {
    f"Best DL ({best_p['model_type']})": y_pred_inv,
    "Transformer": y_pred_trans_inv
}

# Récupération des dates de test alignées
test_dates = df_full['Date'].values[-len(y_test_inv):]

viz.plot_predictions_comparison(y_test_inv, dl_predictions, test_dates, title="Comparaison des Modèles Deep Learning")

## 6. Bilan Comparatif Global (ML vs DL)

Nous chargeons les résultats de Machine Learning et les fusionnons avec nos résultats de Deep Learning pour avoir une vision globale de la performance.

In [ ]:
# Charger les métriques ML
ml_metrics_path = os.path.join(DATA_DIR, 'model_performance_metrics_ML.csv')
if os.path.exists(ml_metrics_path):
    global_results = pd.read_csv(ml_metrics_path)
else:
    global_results = pd.DataFrame(columns=['Model', 'R2', 'RMSE', 'MAE', 'MAPE'])

# Ajouter les métriques DL
best_dl_row = {
    'Model': f"Best DL ({best_p['model_type']})",
    'R2': best_dl_metrics['R2'],
    'RMSE': best_dl_metrics['RMSE'],
    'MAE': best_dl_metrics['MAE'],
    'MAPE': best_dl_metrics['MAPE']
}
trans_row = {
    'Model': 'Transformer',
    'R2': trans_metrics['R2'],
    'RMSE': trans_metrics['RMSE'],
    'MAE': trans_metrics['MAE'],
    'MAPE': trans_metrics['MAPE']
}

global_results = pd.concat([global_results, pd.DataFrame([best_dl_row, trans_row])], ignore_index=True)
global_results = global_results.sort_values(by='R2', ascending=False)

print("--- TABLEAU COMPARATIF FINAL (TOUS MODÈLES) ---")
display(global_results)

# Sauvegarde du bilan final
final_metrics_path = os.path.join(DATA_DIR, 'model_performance_metrics.csv')
global_results.to_csv(final_metrics_path, index=False)
print(f"Bilan final de performance sauvegardé dans : {final_metrics_path}")